In [1]:
from pathlib import Path
import sys
import torch 

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from instancepipelineprior import get_data_loaders
from bayesianprior.discretizedtrain import EarthquakeCNN, evaluate_metrics, logits_to_magnitude

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = PROJECT_ROOT / "bayesianprior" / "data" / "best_model_huberandcrosspoint05.pth"

checkpoint= torch.load(checkpoint_path, map_location=device)
model = EarthquakeCNN().to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()




Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647
Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647


EarthquakeCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(3, 16, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv1d(16, 32, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (4): Sequential(
      (0): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
      (1): ReLU()
      (2): MaxPool1d(kernel_size=2, str

In [2]:
train_loader,val_loader,test_loader = get_data_loaders()


In [4]:
import torch

smoothed_prior = torch.load(
    "data/smoothed_magnitude_prior.pt",
    weights_only=False
)

In [5]:
smoothed_prior = torch.tensor(
    smoothed_prior,
    dtype=torch.float32
)

In [10]:
def logits_to_magnitude_with_prior(logits,bin_centers,prior_probs,alpha):
    prior_probs = prior_probs.to(logits.device)
    bin_centers = bin_centers.to(logits.device)
    log_prior = torch.log(prior_probs.clamp(min=1e-8))

    adjusted_logits = (logits + (alpha - 1.0) * log_prior.unsqueeze(0))
    adjusted_probs = torch.softmax(adjusted_logits,dim=1)
    predictions = torch.sum(adjusted_probs* bin_centers.unsqueeze(0),dim=1)
    return predictions, adjusted_probs

In [6]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [8]:
waveforms, magnitudes = next(
    iter(val_loader)
)

waveforms = waveforms.to(device)
magnitudes = magnitudes.to(device)

print(waveforms.shape)
print(magnitudes.shape)

torch.Size([128, 3, 300])
torch.Size([128])


In [7]:
MIN_MAGNITUDE = 0.0
MAX_MAGNITUDE = 6.6
BIN_WIDTH = 0.1

BIN_CENTERS = torch.arange(
    MIN_MAGNITUDE + BIN_WIDTH / 2,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    dtype=torch.float32
)

In [11]:
logits = model(waveforms)
predictions, probs = logits_to_magnitude_with_prior(logits,BIN_CENTERS,smoothed_prior,alpha=1.0)

In [12]:
predictions, probs 

(tensor([1.6962, 1.8477, 1.8061, 1.2008, 2.5225, 1.5813, 1.5854, 1.3262, 1.4246,
         1.7932, 1.8442, 1.4694, 1.7010, 2.5229, 2.3666, 1.9433, 2.2496, 2.2229,
         2.1298, 1.8962, 2.0890, 1.2621, 1.3179, 1.6173, 1.7870, 1.4955, 1.5712,
         1.3501, 1.7180, 1.7636, 1.0090, 1.1878, 1.3636, 1.4799, 1.7581, 2.0584,
         1.1764, 2.0381, 1.3974, 1.9915, 1.8380, 1.3588, 1.1937, 1.4875, 2.0797,
         1.9125, 1.5858, 1.9827, 2.0927, 1.4705, 1.1512, 2.0373, 1.5428, 1.5700,
         2.2865, 2.1023, 2.1822, 2.6785, 2.1136, 2.2521, 2.2657, 1.8336, 2.3919,
         2.0599, 2.0622, 2.1172, 2.2925, 2.2577, 2.1216, 2.1305, 2.3273, 2.8788,
         3.3600, 2.6403, 2.1569, 2.3544, 2.1014, 2.4964, 2.4817, 2.6511, 3.6917,
         3.0275, 2.2686, 2.0773, 3.4084, 2.6808, 2.5231, 3.5255, 3.4524, 3.0065,
         2.5745, 3.2899, 4.3373, 3.6932, 3.4963, 2.8159, 2.6706, 3.8125, 2.7417,
         2.8053, 3.3982, 3.2426, 3.8420, 2.3980, 2.2852, 3.3417, 3.6005, 2.8896,
         3.3032, 2.1660, 2.1

In [13]:
original_predictions = logits_to_magnitude(
    logits,
    BIN_CENTERS.to(device)
)

print(
    torch.max(
        torch.abs(
            original_predictions - predictions
        )
    )
)

print(
    torch.allclose(
        original_predictions,
        predictions,
        atol=1e-6
    )
)

tensor(0., device='cuda:0', grad_fn=<MaxBackward1>)
True


In [14]:
def evaluate_metrics_with_prior(model,dataloader,device,prior_probs,alpha):
    model.eval()

    all_predictions = []
    all_targets = []

    bin_centers = BIN_CENTERS.to(device)

    with torch.no_grad():

        for waveforms, magnitudes in dataloader:

            waveforms = waveforms.to(device,non_blocking=True)

            magnitudes = magnitudes.to(device,non_blocking=True)

            logits = model(waveforms)

            predictions, _ = (logits_to_magnitude_with_prior(logits,bin_centers,prior_probs,alpha))

            all_predictions.append(
                predictions.cpu()
            )

            all_targets.append(
                magnitudes.cpu()
            )

    predictions = torch.cat(all_predictions)
    targets = torch.cat(all_targets)

    errors = predictions - targets

    mae = torch.abs(errors).mean().item()

    rmse = torch.sqrt(
        torch.mean(errors ** 2)
    ).item()

    bias = errors.mean().item()

    accuracy = (
        torch.abs(errors) <= 0.2
    ).float().mean().item() * 100

    return {
        "alpha": alpha,
        "mae": mae,
        "rmse": rmse,
        "bias": bias,
        "accuracy_0.2": accuracy
    }

In [ ]:
alphas = [
    0.0,
    0.25,
    0.5,
    0.75,
    1.0,
    1.25
]